In [ ]:
import time
import copy
import glob
import random

import numpy as np
import pandas as pd
from PIL import Image

from tqdm.auto import tqdm
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, resnet34

In [ ]:
BASIC_PATH = "/kaggle/input/chest-xray-pneumonia/chest_xray"

In [ ]:
def get_data(data_type: str, is_pneunomial: bool) -> []:
    image_data: str = "NORMAL" if not is_pneunomial else "PNEUMONIA"
    return glob.glob(f"{BASIC_PATH}/{data_type}/{image_data}/*")

In [ ]:
train_normal = get_data("train", False)
train_pneumonia = get_data("train", True)

test_normal = get_data("test", False)
test_pneumonia = get_data("test", True)

train_paths = train_normal + train_pneumonia
test_paths = test_normal + test_pneumonia

In [ ]:
train_labels = [0] * len(train_normal) + [1] * len(train_pneumonia)
test_labels = [0] * len(test_normal) + [1] * len(test_pneumonia)

In [ ]:
train_paths, valid_paths, train_labels, valid_labels = train_test_split(train_paths, train_labels, stratify=train_labels)

In [ ]:
def show_random_images():
    path_random_normal = random.choice(train_normal)
    path_random_pneumonia = random.choice(train_pneumonia)

    fig = plt.figure(figsize=(10, 10))
    
    ax1 = plt.subplot(1, 2, 1)
    ax1.imshow(Image.open(path_random_normal).convert("LA"))
    ax1.set_title("Normal X-ray")
    
    ax2 = plt.subplot(1, 2, 2)
    ax2.imshow(Image.open(path_random_pneumonia).convert("LA"))
    ax2.set_title("Abnormal X-ray")

In [ ]:
show_random_images()

In [ ]:
class PneumoniaXrayDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = self.labels[index]
        label = torch.tensor([label])

        return image, label

In [ ]:
resnet = resnet18()

resnet

In [ ]:
class PneumoniaNet(nn.Module):
    def __init__(self, pretrained=True):
        super(PneumoniaNet, self).__init__()
        self.backbone = resnet18(pretrained=pretrained)
        self.fc = nn.Linear(in_features=512, out_features=1)

    def forward(self, x):
        # Layer 0
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)

        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)

        x = self.backbone.avgpool(x)

        x = x.view(x.size(0), 512)
        x = self.fc(x)

        return x

In [ ]:
image_size = (500, 500)

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(degrees=15),
    transforms.Resize(size=image_size),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(size=image_size),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
train_dataset = PneumoniaXrayDataset(train_paths, train_labels, train_transform)
valid_dataset = PneumoniaXrayDataset(valid_paths, valid_labels, test_transform)

In [ ]:
pretrained = True

model = PneumoniaNet(pretrained=pretrained)
model.cuda()
lr = 3e-3

num_epochs = 5
train_batch_size = 16
valid_batch_size = 16

train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, num_workers=5, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=valid_batch_size, num_workers=5, shuffle=False)

dataloaders = {
    "train": train_dataloader,
    "val": valid_dataloader
}

logging_steps = {
    "train": len(dataloaders["train"]) // 10,
    "val": len(dataloaders["val"]) // 10
}

dataset_sizes = {
    "train": len(train_dataset),
    "val": len(valid_dataset)
}

batch_sizes ={
    "train": train_batch_size,
    "val": valid_batch_size
}

criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=lr)

In [ ]:
def train_model(model, criterion, optiiazer, num_epochs, device="cuda"):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0

    for epoch in tqdm(range(num_epochs), leave=False):
        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for i, (inputs, labels) in tqdm(enumerate(dataloaders[phase]),  total=len(dataloaders[phase]), leave=False):
                inputs = inputs.to(device)
                labels = labels.to(device)
    
                optimizer.zero_grad()
    
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
    
                    preds = outputs.sigmoid() > 0.5
                    loss = criterion(outputs, labels.float())
    
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
    
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)
    
                if (i % logging_steps[phase] == 0) & (i > 0):
                    avg_loss = running_loss / ((i+1) * batch_sizes[phase])
                    avg_acc = running_corrects / ((i+1) * batch_sizes[phase])
    
                    print(f"[{phase}]: {epoch+1} / {num_epochs} | loss: {avg_loss} | acc: {avg_acc}")
    
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
    
            print("{} loss: {:4f} Acc: {:4f}".format(phase, epoch_loss, epoch_acc))
    
            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)

    return model

In [ ]:
model = train_model(model, criterion, optimizer, num_epochs)

In [ ]:
def save_model(model, save_dir="checkpoints", model_name="Model"):
    """
    Save the model state dictionary.
    
    Args:
        model (torch.nn.Module): The model to save.
        save_dir (str): Directory where the model should be saved.
        model_name (str): Name of the model.
    
    Returns:
        str: Path of the saved model file.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"{model_name}_{timestamp}.pth"
    save_path = os.path.join(save_dir, filename)
    
    torch.save(model.state_dict(), save_path)
    
    print(f"Model saved at: {save_path}")
    return save_path

In [ ]:
import os
from datetime import datetime

save_model(model, "trained_model", "PneumoniaModel")

In [ ]:
test_dataset = PneumoniaXrayDataset(test_paths, test_labels, test_transform)

In [ ]:
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, drop_last=False)

In [ ]:
y_pred = []
y_true = []

for i, (tensors, labels) in tqdm(enumerate(test_dataloader), leave=False, total=len(test_dataloader)):
    with torch.no_grad():
        predictions = model(tensors.cuda())  # Get raw logits
        predictions = predictions.sigmoid()  # Apply sigmoid activation
        predictions = (predictions > 0.5).float()  # Convert to binary tensor

        y_pred.append(predictions.cpu())  # Move to CPU
        y_true.append(labels.cpu())       # Move to CPU

In [ ]:
y_pred = [p if isinstance(p, torch.Tensor) else torch.tensor(p) for p in y_pred]
y_true = [t if isinstance(t, torch.Tensor) else torch.tensor(t) for t in y_true]

y_pred = torch.cat(y_pred)
y_true = torch.cat(y_true)

y_pred = y_pred.numpy()
y_true = y_true.numpy()

y_pred = y_pred.astype(np.int64)
y_true = y_true.astype(np.int64)

y_pred = y_pred.reshape(-1)
y_true = y_true.reshape(-1)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
accuracy_score(y_true, y_pred)